In [1]:
from langchain_chroma import Chroma
# Query
def query(vector_store: Chroma, question: str, project: str = None, k: int = 5):
    """Search the vector store and print results."""
    search_kwargs = {"k": k}
    if project:
        search_kwargs["filter"] = {"project": project}
 
    results = vector_store.similarity_search(question, **search_kwargs)
 
    print(f"\nQuery: {question}")
    print(f"Results ({len(results)}):\n")
    for i, doc in enumerate(results, 1):
        print(f"  {i}. [{doc.metadata['source']}]")
        print(f"     {doc.metadata.get('heading_trail', '')}")
        print(f"     {doc.page_content[:150]}...")
        print(f"{'-'*50}")
 
    return results


In [3]:
from langchain_openai import OpenAIEmbeddings
embedding_model = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model = embedding_model)
vector_store = Chroma(
    collection_name = "fastapi",
    embedding_function=embeddings,
    persist_directory="../vectorstore"
)
question = "How to use FastAPI"
query(vector_store, question, project="fastapi", k=10 )


Query: How to use FastAPI
Results (10):

  1. [newsletter.md]
     FastAPI and friends newsletter
     # FastAPI and friends newsletter...
--------------------------------------------------
  2. [first-steps.md]
     First Steps { #first-steps }>Recap, step by step { #recap-step-by-step }>Step 2: create a `FastAPI` "instance" { #step-2-create-a-fastapi-instance }
     ### Step 2: create a `FastAPI` "instance" { #step-2-create-a-fastapi-instance }  
{* ../../docs_src/first_steps/tutorial001_py310.py hl[3] *}  
Here t...
--------------------------------------------------
  3. [fastapi-people.md]
     FastAPI People
     # FastAPI People  
FastAPI has an amazing community that welcomes people from all backgrounds....
--------------------------------------------------
  4. [index.md]
     FastAPI { #fastapi }
     # FastAPI { #fastapi }  
FastAPI framework, high performance, easy to learn, fast to code, ready for production  
---  
**Documentation**: [https://fa...
-----------------------

[Document(id='487b8976921b24047ca9ef939d950850', metadata={'source': 'newsletter.md', 'project': 'fastapi', 'heading_trail': 'FastAPI and friends newsletter'}, page_content='# FastAPI and friends newsletter'),
 Document(id='7c5ce0dcd9690a340cfac2271a04cfdd', metadata={'source': 'first-steps.md', 'project': 'fastapi', 'heading_trail': 'First Steps { #first-steps }>Recap, step by step { #recap-step-by-step }>Step 2: create a `FastAPI` "instance" { #step-2-create-a-fastapi-instance }'}, page_content='### Step 2: create a `FastAPI` "instance" { #step-2-create-a-fastapi-instance }  \n{* ../../docs_src/first_steps/tutorial001_py310.py hl[3] *}  \nHere the `app` variable will be an "instance" of the class `FastAPI`.  \nThis will be the main point of interaction to create all your API.'),
 Document(id='e787754e80096bdcf46b997f5fc217c5', metadata={'source': 'fastapi-people.md', 'project': 'fastapi', 'heading_trail': 'FastAPI People'}, page_content='# FastAPI People  \nFastAPI has an amazing com

In [ ]:
# Check if ChromaDB has any data
import chromadb

client = chromadb.PersistentClient(path="../vectorstore")
collection = client.get_collection("fastapi")
print(f"Total chunks stored: {collection.count()}")

In [4]:
# Eval... 
# dataset

eval_set = [
    {
        "question": "What is FastAPI?",
        "expected_sources": ["index.md"],
        "expected_keywords": ["framework", "python", "api", "high-performance"],
    },
    {
        "question": "How to use OAuth2 with JWT in FastAPI?",
        "expected_sources": ["oauth2-jwt.md"],
        "expected_keywords": ["token", "jwt", "password", "bearer"],
    },
    {
        "question": "How to deploy FastAPI with Docker?",
        "expected_sources": ["docker.md"],
        "expected_keywords": ["docker", "container", "dockerfile"],
    },
    {
        "question": "How to handle path parameters?",
        "expected_sources": ["path-params.md"],
        "expected_keywords": ["path", "parameter", "type"],
    },
    {
        "question": "How to test a FastAPI application?",
        "expected_sources": ["testing.md"],
        "expected_keywords": ["test", "testclient", "pytest"],
    },
]

In [5]:
# Eval... retrieval quality

def evaluate_retrieval(vector_store, eval_set, project="fastapi", k=5):
    results = {
        "total": len(eval_set),
        "source_hits": 0,
        "keyword_hits": 0,
        "details": [],
    }

    for test in eval_set:
        retrieved = vector_store.similarity_search(
            test["question"], k=k, filter={"project": project}
        )

        # Check 1: Did the right source file appear in top-k?
        retrieved_sources = [doc.metadata["source"] for doc in retrieved]
        source_found = any(
            expected in retrieved_sources
            for expected in test["expected_sources"]
        )

        # Check 2: Do retrieved chunks contain expected keywords?
        all_text = " ".join(doc.page_content.lower() for doc in retrieved)
        keywords_found = sum(
            1 for kw in test["expected_keywords"]
            if kw.lower() in all_text
        )
        keyword_score = keywords_found / len(test["expected_keywords"])

        results["source_hits"] += int(source_found)
        results["keyword_hits"] += keyword_score

        results["details"].append({
            "question": test["question"],
            "source_match": source_found,
            "keyword_score": f"{keyword_score:.0%}",
            "top_sources": retrieved_sources[:3],
        })

    # Summary
    source_accuracy = results["source_hits"] / results["total"]
    keyword_accuracy = results["keyword_hits"] / results["total"]

    print(f"\n{'='*50}")
    print(f"  RETRIEVAL EVALUATION")
    print(f"  Source accuracy:  {source_accuracy:.0%}")
    print(f"  Keyword accuracy: {keyword_accuracy:.0%}")
    print(f"{'='*50}\n")

    for d in results["details"]:
        status = "✓" if d["source_match"] else "✗"
        print(f"  {status} {d['question']}")
        print(f"    Keywords: {d['keyword_score']}")
        print(f"    Got: {d['top_sources']}")
        print()

    return results


# Run it
evaluate_retrieval(vector_store, eval_set)


  RETRIEVAL EVALUATION
  Source accuracy:  100%
  Keyword accuracy: 100%

  ✓ What is FastAPI?
    Keywords: 100%
    Got: ['newsletter.md', 'index.md', 'fastapi-people.md']

  ✓ How to use OAuth2 with JWT in FastAPI?
    Keywords: 100%
    Got: ['index.md', 'first-steps.md', 'oauth2-scopes.md']

  ✓ How to deploy FastAPI with Docker?
    Keywords: 100%
    Got: ['docker.md', 'index.md', 'first-steps.md']

  ✓ How to handle path parameters?
    Keywords: 100%
    Got: ['path-params.md', 'path-params.md', 'path-params.md']

  ✓ How to test a FastAPI application?
    Keywords: 100%
    Got: ['testing.md', 'testing.md', 'testclient.md']



{'total': 5,
 'source_hits': 5,
 'keyword_hits': 5.0,
 'details': [{'question': 'What is FastAPI?',
   'source_match': True,
   'keyword_score': '100%',
   'top_sources': ['newsletter.md', 'index.md', 'fastapi-people.md']},
  {'question': 'How to use OAuth2 with JWT in FastAPI?',
   'source_match': True,
   'keyword_score': '100%',
   'top_sources': ['index.md', 'first-steps.md', 'oauth2-scopes.md']},
  {'question': 'How to deploy FastAPI with Docker?',
   'source_match': True,
   'keyword_score': '100%',
   'top_sources': ['docker.md', 'index.md', 'first-steps.md']},
  {'question': 'How to handle path parameters?',
   'source_match': True,
   'keyword_score': '100%',
   'top_sources': ['path-params.md', 'path-params.md', 'path-params.md']},
  {'question': 'How to test a FastAPI application?',
   'source_match': True,
   'keyword_score': '100%',
   'top_sources': ['testing.md', 'testing.md', 'testclient.md']}]}

In [7]:
# Trying similarity search with threshold

def search_with_threshold(vector_store, question, project=None, k=5, min_similarity=0.75):
    """Only return chunks above the similarity threshold."""

    filter_dict = {"project": project} if project else None

    # This returns (Document, distance) pairs
    results_with_scores = vector_store.similarity_search_with_score(
        question, k=k, filter=filter_dict
    )

    # Convert distance to similarity and filter
    filtered = []
    for doc, distance in results_with_scores:
        similarity = 1 - distance
        doc.metadata["similarity"] = round(similarity, 4)

        if similarity >= min_similarity:
            filtered.append(doc)

    return filtered


# Test it
results = search_with_threshold(
    vector_store,
    "How to use OAuth2 with JWT?",
    project="fastapi",
    min_similarity=0.75,
)

print(f"Results above 0.75: {len(results)}\n")
for doc in results:
    sim = doc.metadata["similarity"]
    src = doc.metadata["source"]
    print(f"  [{sim:.4f}] {src}: {doc.page_content[:80]}...")

Results above 0.75: 0



In [12]:
# Eval... retrieval quality

def evaluate_threshold_retrieval(
    vector_store, eval_set, project="fastapi", k=5, min_similarity=0.75
):
    results = {
        "total": len(eval_set),
        "source_hits": 0,
        "keyword_hits": 0,
        "avg_similarity": 0,
        "avg_results_returned": 0,
        "fallback_count": 0,
        "details": [],
    }

    all_similarities = []

    for test in eval_set:
        retrieved = search_with_threshold(
            vector_store, test["question"],
            project=project, k=k, min_similarity=min_similarity,
        )

        # Check 1: Source match
        retrieved_sources = [doc.metadata["source"] for doc in retrieved]
        source_found = any(
            expected in retrieved_sources
            for expected in test["expected_sources"]
        )

        # Check 2: Keyword match
        all_text = " ".join(doc.page_content.lower() for doc in retrieved)
        keywords_found = sum(
            1 for kw in test["expected_keywords"]
            if kw.lower() in all_text
        )
        keyword_score = keywords_found / len(test["expected_keywords"])

        # Check 3: Similarity scores
        similarities = [doc.metadata.get("similarity", 0) for doc in retrieved]
        avg_sim = sum(similarities) / len(similarities) if similarities else 0
        all_similarities.extend(similarities)

        # Check 4: Did fallback trigger?
        used_fallback = any(doc.metadata.get("below_threshold") for doc in retrieved)

        results["source_hits"] += int(source_found)
        results["keyword_hits"] += keyword_score
        results["avg_results_returned"] += len(retrieved)
        results["fallback_count"] += int(used_fallback)

        results["details"].append({
            "question": test["question"],
            "source_match": source_found,
            "keyword_score": f"{keyword_score:.0%}",
            "results_returned": len(retrieved),
            "similarities": [f"{s:.4f}" for s in similarities],
            "avg_similarity": f"{avg_sim:.4f}",
            "used_fallback": used_fallback,
            "top_sources": retrieved_sources[:3],
        })

    # Summary
    total = results["total"]
    source_accuracy = results["source_hits"] / total
    keyword_accuracy = results["keyword_hits"] / total
    avg_returned = results["avg_results_returned"] / total
    avg_sim = sum(all_similarities) / len(all_similarities) if all_similarities else 0

    print(f"\n{'='*55}")
    print(f"  THRESHOLD RETRIEVAL EVALUATION (min={min_similarity})")
    print(f"{'='*55}")
    print(f"  Source accuracy:     {source_accuracy:.0%}")
    print(f"  Keyword accuracy:    {keyword_accuracy:.0%}")
    print(f"  Avg similarity:      {avg_sim:.4f}")
    print(f"  Avg results/query:   {avg_returned:.1f} / {k}")
    print(f"  Fallback triggered:  {results['fallback_count']} / {total}")
    print(f"{'='*55}\n")

    for d in results["details"]:
        status = "✓" if d["source_match"] else "✗"
        fallback = " (FALLBACK)" if d["used_fallback"] else ""
        print(f"  {status} {d['question']}{fallback}")
        print(f"    Keywords:    {d['keyword_score']}")
        print(f"    Returned:    {d['results_returned']} chunks")
        print(f"    Avg sim:     {d['avg_similarity']}")
        print(f"    Scores:      {d['similarities']}")
        print(f"    Sources:     {d['top_sources']}")
        print()

    return results


# Run at different thresholds to find the sweet spot
for threshold in [0.30, 0.35, 0.40, 0.42]:
    evaluate_threshold_retrieval(
        vector_store, eval_set, min_similarity=threshold
    )


  THRESHOLD RETRIEVAL EVALUATION (min=0.3)
  Source accuracy:     60%
  Keyword accuracy:    70%
  Avg similarity:      0.3710
  Avg results/query:   1.8 / 5
  Fallback triggered:  0 / 5

  ✓ What is FastAPI?
    Keywords:    100%
    Returned:    5 chunks
    Avg sim:     0.3734
    Scores:      ['0.4148', '0.4120', '0.3917', '0.3451', '0.3032']
    Sources:     ['newsletter.md', 'index.md', 'fastapi-people.md']

  ✗ How to use OAuth2 with JWT in FastAPI?
    Keywords:    50%
    Returned:    1 chunks
    Avg sim:     0.3493
    Scores:      ['0.3493']
    Sources:     ['index.md']

  ✓ How to deploy FastAPI with Docker?
    Keywords:    100%
    Returned:    2 chunks
    Avg sim:     0.3751
    Scores:      ['0.3988', '0.3514']
    Sources:     ['docker.md', 'index.md']

  ✗ How to handle path parameters?
    Keywords:    0%
    Returned:    0 chunks
    Avg sim:     0.0000
    Scores:      []
    Sources:     []

  ✓ How to test a FastAPI application?
    Keywords:    100%
    Retu

In [11]:
# Debug: see what ChromaDB is actually returning
results_with_scores = vector_store.similarity_search_with_score(
    "What is FastAPI?", k=5, filter={"project": "fastapi"}
)

for doc, score in results_with_scores:
    print(f"  Raw score: {score:.6f}  |  1-score: {1-score:.6f}  |  {doc.metadata['source']}")

  Raw score: 0.585181  |  1-score: 0.414819  |  newsletter.md
  Raw score: 0.587968  |  1-score: 0.412032  |  index.md
  Raw score: 0.608272  |  1-score: 0.391728  |  fastapi-people.md
  Raw score: 0.654923  |  1-score: 0.345077  |  cloud.md
  Raw score: 0.696821  |  1-score: 0.303179  |  index.md
